<a href="https://colab.research.google.com/github/yaelglz/modelos-estocasticos-toma-decisiones/blob/main/proyecto/prediccion_del_clima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto: Predicciones del Estado del Tiempo en Seúl, Corea del Sur

Este notebook presenta un estudio del clima en Seúl durante el mes de marzo, aplicando modelos estocásticos para predecir el comportamiento atmosférico.

## 1. Datos del problema y planteamiento del problema
Seúl, la capital de Corea del Sur, experimenta la transición del invierno a la primavera durante el mes de marzo. El clima suele ser variable, con días soleados que comienzan a calentar el ambiente, intercalados con días nublados y algunas precipitaciones ligeras.

Para este análisis, se han recolectado datos históricos del estado del tiempo día tras día durante un mes de marzo típico (basado en el calendario observado), identificando tres estados principales:
*   **s1: Soleado (S)**
*   **s2: Nublado (N)** (incluye nubes parciales y nublado total)
*   **s3: Lluvioso (LL)**

Asumiremos que las condiciones del clima forman una **Cadena de Markov**, donde el clima de mañana depende únicamente del clima de hoy.

<div style="text-align: center;">
    <img src="clima_seul.png" width="500">
    <p style="text-align: center;"><em>Figura 1. Representación del estado del clima en Seúl durante marzo.</em></p>
</div>

A continuación, importamos las librerías necesarias y definimos la secuencia de estados observada en el calendario de marzo para Seúl.

In [23]:
import numpy as np
import pandas as pd

# Secuencia de estados observada en marzo (31 días)
# S = Soleado (s1), N = Nublado (s2), LL = Lluvioso (s3)
datos_marzo = [
    'LL', 'S', 'N', 'N', 'N', 'N', 'N', 'N', 'S', 'N', 
    'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'S', 'S', 
    'S', 'N', 'N', 'N', 'N', 'S', 'N', 'S', 'N', 'N', 'S'
]

print(f"Total de días analizados: {len(datos_marzo)}")
print(f"Secuencia: {datos_marzo}")


Total de días analizados: 31
Secuencia: ['LL', 'S', 'N', 'N', 'N', 'N', 'N', 'N', 'S', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'S', 'S', 'S', 'N', 'N', 'N', 'N', 'S', 'N', 'S', 'N', 'N', 'S']


## 2. Cálculos del modelo estocástico

Para construir el modelo, debemos contar cuántas veces se pasa de un estado $s_i$ a un estado $s_j$. La probabilidad de transición se define como:
$$p_{i,j} = P(X_{n+1} = s_j | X_n = s_i)$$

El siguiente bloque de código calcula las frecuencias de transición y prepara la estructura de la matriz.


In [24]:
estados = ['S', 'N', 'LL']
n_estados = len(estados)
matriz_conteo = np.zeros((n_estados, n_estados))

mapa_estados = {'S': 0, 'N': 1, 'LL': 2}

# Contar transiciones
for i in range(len(datos_marzo) - 1):
    actual = mapa_estados[datos_marzo[i]]
    siguiente = mapa_estados[datos_marzo[i+1]]
    matriz_conteo[actual][siguiente] += 1

print("Matriz de Conteo de Transiciones:")
df_conteo = pd.DataFrame(matriz_conteo, index=estados, columns=estados)
print(df_conteo)


Matriz de Conteo de Transiciones:
      S     N   LL
S   2.0   5.0  0.0
N   5.0  17.0  0.0
LL  1.0   0.0  0.0


## 3. Calculo de las probabilidades de transición y matriz de transición a partir de los datos

Normalizamos las filas de nuestra matriz de conteo para obtener las probabilidades de transición.


In [25]:
# Calcular probabilidades (normalizar por filas)
sumas_filas = matriz_conteo.sum(axis=1, keepdims=True)
P = np.divide(matriz_conteo, sumas_filas, out=np.zeros_like(matriz_conteo), where=sumas_filas!=0)

# Convertir a DataFrame para mejor visualización
df_P = pd.DataFrame(P, index=estados, columns=estados)
print("Matriz de Transición P:")
print(df_P)


Matriz de Transición P:
           S         N   LL
S   0.285714  0.714286  0.0
N   0.227273  0.772727  0.0
LL  1.000000  0.000000  0.0


### Explicación de la Matriz de Transición
Cada entrada $p_{i,j}$ representa la probabilidad de que, dado que hoy el clima está en el estado $i$, mañana esté en el estado $j$.
*   **P[S, S]**: Probabilidad de que si hoy está soleado, mañana también lo esté.
*   **P[N, S]**: Probabilidad de que si hoy está nublado, mañana despeje y esté soleado.
*   **Fila suma 1**: La suma de cada fila en la matriz de transición siempre debe ser igual a 1.


## 4. Calculo de las probabilidades de expresiones del estado del clima

### 4.1. Vector de Probabilidad Inicial ($v$)
El vector $v$ representa el estado actual del clima. Si hoy es el último día del mes observado (31 de marzo) y fue Soleado, el vector inicial es $v = [1, 0, 0]$.


In [26]:
# Suponiendo que el último día observado fue Soleado (S)
v = np.array([1, 0, 0]) 
print(f"Vector de estado inicial (Hoy es Soleado): {v}")


Vector de estado inicial (Hoy es Soleado): [1 0 0]


### 4.2. La probabilidad de que en un día el clima exprese el estado si, es decir, $vP$.


In [27]:
v_1_dia = np.dot(v, P)
print(f"Probabilidades para mañana: {dict(zip(estados, v_1_dia))}")


Probabilidades para mañana: {'S': 0.2857142857142857, 'N': 0.7142857142857143, 'LL': 0.0}


### 4.3. La probabilidad de que en dos días el clima exprese el estado si, es decir, $vP^2$


In [28]:
v_2_dias = np.dot(v, np.linalg.matrix_power(P, 2))
print(f"Probabilidades para pasado mañana: {dict(zip(estados, v_2_dias))}")


Probabilidades para pasado mañana: {'S': 0.2439703153988868, 'N': 0.7560296846011132, 'LL': 0.0}


### 4.4. Vector de Probabilidades Estacionarias (Distribución de Largo Plazo $\pi$)

El vector estacionario, denotado como $\pi$ (letra griega "pi"), representa el **punto de equilibrio** de nuestro sistema. Es como responder a la pregunta: *"Si observamos el clima de Seúl durante muchos años en marzo, ¿cuál es el porcentaje real de días soleados, nublados o lluviosos?"*.

A diferencia de las predicciones a corto plazo, el vector estacionario nos muestra la **tendencia natural** del clima de la región.

**¿Cómo se calcula matemáticamente?**
Buscamos un vector $\pi$ que cumpla con la condición:
$$\pi P = \pi$$
Esto significa que si multiplicamos el vector por la matriz de transición, el resultado es el mismo vector (el sistema ya no cambia, está en equilibrio). Además, como son probabilidades, la suma de sus componentes debe ser igual a 1:
$$\sum \pi_i = 1$$


#### Paso 1: Preparación del sistema (Matriz $P^T - I$)
Para resolver $\pi P = \pi$, lo transformamos en un sistema de ecuaciones igual a cero: $\pi(P - I) = 0$. 
Debido a cómo Python resuelve sistemas lineales ($Ax = b$), trabajamos con la transpuesta de $P$.

Primero, restamos una **matriz identidad** (unos en la diagonal, ceros fuera) a la transpuesta de nuestra matriz de transición $P$.

In [29]:
# 1. Matriz Transpuesta de P
Pt = P.T
print("1. Matriz Transpuesta de P (P.T):")
print(pd.DataFrame(Pt, index=estados, columns=estados))

# 2. Matriz Identidad (I)
I = np.eye(n_estados)
print("\n2. Matriz Identidad (I):")
print(I)

# 3. Matriz A inicial (A = P.T - I)
A = Pt - I
print("\n3. Matriz A inicial (P.T - I):")
print(pd.DataFrame(A, index=estados, columns=estados))


1. Matriz Transpuesta de P (P.T):
           S         N   LL
S   0.285714  0.227273  1.0
N   0.714286  0.772727  0.0
LL  0.000000  0.000000  0.0

2. Matriz Identidad (I):
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

3. Matriz A inicial (P.T - I):
           S         N   LL
S  -0.714286  0.227273  1.0
N   0.714286 -0.227273  0.0
LL  0.000000  0.000000 -1.0


#### Paso 2: Condición de Normalización (Suma = 1)
El sistema anterior tiene infinitas soluciones. Para encontrar la correcta, debemos forzar que la suma de todas las probabilidades sea **1** (100%).
Reemplazamos la última fila de nuestra matriz $A$ por una fila de puros **unos**.

In [30]:
# Reemplazamos la última fila de A
A[-1] = np.ones(n_estados)

print("Matriz A final (con la condición de suma = 1 en la última fila):")
print(pd.DataFrame(A, index=estados, columns=estados))


Matriz A final (con la condición de suma = 1 en la última fila):
           S         N   LL
S  -0.714286  0.227273  1.0
N   0.714286 -0.227273  0.0
LL  1.000000  1.000000  1.0


#### Paso 3: Definición del vector de resultados ($b$)
Queremos que las primeras ecuaciones den $0$ y la última (la de la suma) dé $1$. Creamos un vector $b$ con ceros y un uno al final.

In [31]:
# Vector b: [0, 0, 1]
b = np.zeros(n_estados)
b[-1] = 1

print("Vector de resultados b:")
print(b)


Vector de resultados b:
[0. 0. 1.]


#### Paso 4: Resolución del Sistema y Resultado Final
Finalmente, resolvemos el sistema de ecuaciones $A\pi = b$ para obtener los valores de nuestro vector estacionario.

In [32]:
# Resolvemos el sistema de ecuaciones
pi = np.linalg.solve(A, b)

print("Vector Estacionario (Distribución a largo plazo):")
for estado, prob in zip(estados, pi):
    print(f"- {estado}: {prob:.2%}")


Vector Estacionario (Distribución a largo plazo):
- S: 24.14%
- N: 75.86%
- LL: 0.00%


## 5. Conclusiones

A partir de los datos históricos de marzo en Seúl:
1.  **Dominancia del clima nublado**: El vector estacionario muestra que la mayor parte del tiempo el clima es nublado (Estado $s_2$), lo cual es consistente con la temporada de transición en Corea.
2.  **Persistencia**: La matriz de transición revela una alta probabilidad de que un día nublado sea seguido por otro nublado.
3.  **Análisis Predictivo**: El modelo permite estimar el estado del clima a corto plazo y su distribución a largo plazo, proporcionando una herramienta estocástica útil para la toma de decisiones basada en datos históricos.
